<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 📧 综合案例：邮件 Agent

**一句话定位**：把 node 风格、wrap 风格、HITL 三类 middleware 揉进一个真实场景——先登录才能用收发邮件的工具。

</div>

本例的 agent：
- **先认证用户** —— 只有 state 里的邮箱/密码能匹配到硬编码值，才允许进"收件箱"
- **动态开放工具和提示词** —— 是否已认证这个条件决定了 `dynamic_tool_call` / `dynamic_prompt` 给出什么
- **检查收件箱** —— 用一个只读工具
- **发邮件** —— 挂了 human-in-the-loop，发送前要人批准

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

## 🛠️ 为什么 `authenticate` 工具要返回 `Command`？

正常情况下，工具只是**返回一个值**（通常是字符串），LangChain 会自动把它包进 `ToolMessage` 并追加到 `messages[]`。这是工具默认唯一能做的事：**产出一个结果让 LLM 下一步去读。**

但 `authenticate` 需要做更多——它必须**写入 agent 的 state**，把 `authenticated` 从 `False` 翻成 `True`。普通的返回值**碰不到 state**，只能加消息。所以如果工具只是返回 `"Successfully authenticated"`，LLM 会以为认证成功了，但 `AuthenticatedState` 里的 `authenticated` 字段其实没变——而我们的 `dynamic_tool_call` / `dynamic_prompt` middleware 恰恰是靠 `request.state.get("authenticated")` 来决定要不要解锁收件箱工具的，于是永远解锁不了。🔒

### ✅ `Command` = "既返回结果，又更新 state"

`Command(update={...})` 就是让工具一次性做到这两件事的办法：

```python
return Command(update={
    "authenticated": True,                       # 1️⃣ 写入 agent state
    "messages": [ToolMessage(                     # 2️⃣ 仍然要回话给 LLM
        "Successfully authenticated",
        tool_call_id=runtime.tool_call_id)]
})
```

| 返回什么 | 能做到什么 | 改 state 吗？ |
|---|---|---|
| 字符串 / 普通值 | 给 LLM 加一条 `ToolMessage` | ❌ 不能 |
| `Command(update={...})` | 加消息 **同时** 修改 state | ✅ 能 |

### 🔑 两个容易忽略的细节

- **`ToolMessage` 要自己手动构造。** 一旦返回 `Command`，LangChain 就不会再自动帮你包返回值了，所以要自己传 `tool_call_id=runtime.tool_call_id`，这样这条消息才能正确对应上触发它的那次工具调用。
- **这正是认证门禁真正生效的原因。** `authenticated` 这次写入会在**下一轮** LLM 调用时被 middleware 读到，从而切换成 `[check_inbox, send_email]` 和认证后的提示词。工具写 state → 下游行为跟着变。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**：对比本 notebook 后面出现的另一个 `Command`——`Command(resume={...})`。那个不是工具返回的，而是传进 `agent.invoke()` 去**恢复**一次暂停的 human-in-the-loop 中断。同一个类，两种完全不同的用法：`update=` 是工具从内部写 state；`resume=` 是把人的决策喂回一个暂停中的 agent。

</div>

---

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

> 注：录制后这里的提示词做了调整，让模型更稳定地匹配录制时的流程。你仍可能得到和录制不同的响应，这是预期的——可能需要调整 human message 来引导模型给出合适的回复。

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

I checked your inbox. Here’s the message I found:

From: Jane (jane@example.com)
Subject: (not shown)
Hello Julie,
Hi Julie, I'm going to be in town next week and was wondering if we could grab a coffee?
- best, Jane

Would you like me to reply and suggest a few times? Here are a few draft options you can choose from (or I can tailor them to your style):

Option 1 — casual and friendly
Subject: Re: Coffee next week
Hi Jane,
That sounds great! I’d love to. I’m free Tuesday or Thursday afternoon next week. Do either work for you? If you have a preferred cafe, let me know.
Looking forward to it!

Option 2 — specific times
Subject: Re: Coffee next week
Hi Jane,
I’d love to meet up. How about Tuesday at 3:00 PM or Thursday at 11:00 AM? If neither works, I’m happy to adapt. Let me know what’s convenient.

Option 3 — location suggestion
Subject: Re: Coffee next week
Hi Jane,
I’d love to catch up. Want to meet at [Cafe Name] next week? I’m free Tue and Thu afternoon. What works for you?

If yo

In [9]:

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [10]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,
That sounds great! I’d love to. I’m free Tuesday or Thursday afternoon next week. Do either work for you? If you have a preferred cafe, let me know.
Looking forward to it!


In [11]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Email sent to jane@example.com with subject "Re: Coffee next week" and body:
Hi Jane,
That sounds great! I’d love to. I’m free Tuesday or Thursday afternoon next week. Do either work for you? If you have a preferred cafe, let me know.
Looking forward to it!


In [12]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='cd6784ff-58d9-400d-bb0e-3da84fc0dc46'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 286, 'prompt_tokens': 151, 'total_tokens': 437, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DOoSKZcepxV1pNVbCpkTQRJqWnZzV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d3ab1-5dbd-7223-9e59-5edf45a39d77-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_nGaIRLsjvsIr8BHfSN4dxN0k', 'type': 'tool_call